In [ ]:
QUEUE = "Q2Q3Q4"  # Q1 or Q2Q3Q4
MODE = "full"  # timing or full
CODE_DATASET = "ds-code1"  # preferred Kaggle input slug
GPU_H_USED = 2.2049  # Logged through Q1. Add extra GPU session time if applicable.
Q1_RESUME = False  # Recovery from the failed Q1 is complete.
assert QUEUE in ("Q1","Q2Q3Q4") and MODE in ("timing","full")
import time, os, subprocess, sys, json, hashlib, shutil, zipfile
from pathlib import Path
SESSION_START = time.time()


In [ ]:
probe = subprocess.check_output(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],text=True)
print(probe); subprocess.run(["df","-h"],check=True)
assert len(probe.strip().splitlines()) == 2 and all("T4" in x for x in probe.strip().splitlines()), "Select GPU T4 x2"
from urllib.request import urlopen
try:
    with urlopen("https://huggingface.co",timeout=30) as response: assert response.status == 200
except Exception as error: raise RuntimeError("Internet unavailable: enable Internet and complete Kaggle verification") from error


In [ ]:
def safe_extract(archive, destination):
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            if not (destination / item.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError(f"Unsafe ZIP path: {item.filename}")
        z.extractall(destination)


def stage_code_bundle(input_root, destination, preferred_slug):
    inventory_path = Path("common/configs/bundle-inventory.json")
    archives = []
    unpacked = []
    zip_names = []
    for directory, _, filenames in os.walk(input_root):
        folder = Path(directory)
        if folder.name == "configs" and inventory_path.name in filenames:
            bundle_root = folder.parent.parent
            if (bundle_root / inventory_path).is_file():
                unpacked.append(bundle_root)
        for name in filenames:
            if not name.lower().endswith(".zip"):
                continue
            candidate = folder / name
            zip_names.append(str(candidate.relative_to(input_root)))
            try:
                with zipfile.ZipFile(candidate) as z:
                    z.getinfo(inventory_path.as_posix())
            except (KeyError, zipfile.BadZipFile):
                continue
            archives.append(candidate)

    candidates = archives + unpacked
    preferred = [p for p in candidates if preferred_slug.casefold() in
                 {part.casefold() for part in p.relative_to(input_root).parts}]
    candidates = preferred or candidates
    if len(candidates) > 1:
        options = "\n".join(str(p.relative_to(input_root)) for p in sorted(candidates))
        raise RuntimeError(f"Found multiple project bundles. Attach just one or set CODE_DATASET to its slug.\n{options}")
    if not candidates:
        mounts = sorted(p.name for p in input_root.iterdir()) if input_root.is_dir() else []
        raise FileNotFoundError(
            f"Attach the private {preferred_slug} dataset containing the DS project bundle using Add Input. "
            f"Visible input folders: {mounts}. ZIPs found: {sorted(zip_names)[:10]}. "
            f"A valid bundle contains {inventory_path.as_posix()}."
        )

    bundle = candidates[0]
    destination.mkdir(parents=True, exist_ok=True)
    if bundle in archives:
        safe_extract(bundle, destination)
    else:
        inventory = json.loads((bundle / inventory_path).read_text(encoding="utf-8"))
        for entry in inventory["files"]:
            relative = Path(entry["path"])
            source = bundle / relative
            target = destination / relative
            if not source.resolve().is_relative_to(bundle.resolve()) or not target.resolve().is_relative_to(destination.resolve()):
                raise ValueError(f"Unsafe inventory path: {relative}")
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source, target)
        shutil.copyfile(bundle / inventory_path, destination / inventory_path)

    inventory = json.loads((destination / inventory_path).read_text(encoding="utf-8"))
    for entry in inventory["files"]:
        path = destination / entry["path"]
        if not path.resolve().is_relative_to(destination.resolve()):
            raise ValueError(f"Unsafe inventory path: {entry['path']}")
        if hashlib.sha256(path.read_bytes()).hexdigest() != entry["sha256"]:
            raise ValueError(f"Code bundle hash mismatch: {entry['path']}")
    print("Verified code bundle:", bundle)
    return inventory


def restore_q1_output(input_root, output, archive_sha, tree_sha):
    # A saved Kaggle notebook exposes the verified ZIP alongside its loose output.
    archives = sorted(input_root.rglob('results_Q1.zip'))
    if archives:
        for archive in archives:
            with archive.open('rb') as f:
                matches = hashlib.file_digest(f, 'sha256').hexdigest() == archive_sha
            if matches:
                safe_extract(archive, output)
                print('Restored the verified 270 Q1 results and source checkpoints from:', archive)
                return
        raise ValueError('No Q1 ZIP matched the verified checksum. Attach the successful Q1 notebook output version.')

    extracted = [p.parent for p in input_root.rglob('session_Q1.json') if (p.parent / 'raw').is_dir()]
    if len(extracted) != 1:
        raise RuntimeError('In Add Input, attach the successful Q1 Notebook Output containing out/results_Q1.zip.')
    source = extracted[0]
    files = sorted((p for p in source.rglob('*') if p.is_file()), key=lambda p: p.relative_to(source).as_posix())
    tree = hashlib.sha256()
    for path in files:
        relative = path.relative_to(source)
        if not path.resolve().is_relative_to(source.resolve()) or not (output / relative).resolve().is_relative_to(output.resolve()):
            raise ValueError(f'Unsafe Q1 input path: {path}')
        with path.open('rb') as f:
            checksum = hashlib.file_digest(f, 'sha256').hexdigest()
        tree.update((relative.as_posix() + '\0' + checksum + '\n').encode())
    if tree.hexdigest() != tree_sha:
        raise ValueError('Extracted Q1 input differs from the verified archive. Attach the successful Q1 notebook output version.')
    for path in files:
        target = output / path.relative_to(source)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(path, target)
    print('Restored the verified 270 Q1 results and source checkpoints from:', source)


ROOT = Path("/kaggle/working/DS")
inv = stage_code_bundle(Path("/kaggle/input"), ROOT, CODE_DATASET)
expected_fingerprint = "6216f8af01b6d87d88192a1ccd9035f248b79e9d5ffd8f654dde254bfea0e1aa"
if json.loads((ROOT / "docs/freeze.json").read_text())["fingerprint"] != expected_fingerprint:
    raise RuntimeError("Use a bundle built by python common/bundle. For an exact replay, use the archived notebook and frozen bundle.")
OUT=Path('/kaggle/working/out');OUT.mkdir(exist_ok=True)
if QUEUE == 'Q2Q3Q4':
    restore_q1_output(Path('/kaggle/input'), OUT,
                      '4e4460af2c684cde951dcf46e7491e437fb9f67126a27130e7311562d29d3fef',
                      'eafcda2e62668f17e968ef97182aeab2bb0f9476ee1f1c6314c01e040e7fb94d')
    if json.loads((OUT/'session_Q1.json').read_text())['freeze']['fingerprint'] != expected_fingerprint:
        raise ValueError('Q1 belongs to the archived layout. Use the archived notebook and frozen code for an exact replay.')
    for worker in (OUT/'shard0',OUT/'shard1'):
        (worker/'checkpoints').mkdir(parents=True,exist_ok=True)
        for cp in list(OUT.glob('shard*/checkpoints/*.pt')):
            if not (worker/'checkpoints'/cp.name).exists(): shutil.copy2(cp,worker/'checkpoints'/cp.name)
os.environ['DS_SOURCE_COMMIT']=inv['git_commit']
# Match the verified torch/torchvision API pair; install CPU-side before workers start.
subprocess.run([sys.executable,'-m','pip','install','--quiet','torch==2.11.0','torchvision==0.26.0','--index-url','https://download.pytorch.org/whl/cu128'],check=True)
subprocess.run([sys.executable,'-m','pip','install','--quiet','numpy==2.3.3','pandas==2.3.2','scipy==1.16.2','scikit-learn==1.8.0','Pillow==11.3.0','PyYAML==6.0.3','huggingface_hub==2.4.0'],check=True)
TEMP=Path('/kaggle/temp'); TEMP.mkdir(exist_ok=True)
os.environ.update(DATA_DIR=str(TEMP/'robustad'),CACHE_DIR=str(TEMP/'features'),OUT_DIR=str(OUT),QUEUE_BATCH=QUEUE)


In [ ]:
from huggingface_hub import snapshot_download
cfg=json.loads((ROOT/'common/configs/config.yaml').read_text()); groups=['MetalParts']+(['PCB'] if cfg['pcb_go_conditions'] else [])
listing=inv['dataset_listing']; files=[x for x in listing['files'] if x['rfilename'].split('/')[0] in groups]
assert listing['revision']==cfg['dataset_revision'] and 'license: cc-by-4.0' in listing['card']
assert shutil.disk_usage(TEMP).free > sum(x['size'] for x in files)+15*1024**3, 'Insufficient temporary disk space'
snapshot_download('AmazonScience/RobustAD',repo_type='dataset',revision=cfg['dataset_revision'],allow_patterns=[g+'/**' for g in groups]+['README.md'],local_dir=TEMP/'robustad',max_workers=4)
raw=TEMP/'robustad'
assert (raw/'README.md').read_text()==listing['card'], 'Pinned dataset card mismatch'
assert len([p for g in groups for p in (raw/g).rglob('*') if p.is_file()])==len(files), 'Dataset file count mismatch'
for item in files:
    p=raw/item['rfilename']; assert p.stat().st_size==item['size'], str(p)
    with p.open('rb') as f:
        if item.get('lfs'): actual=hashlib.file_digest(f,'sha256').hexdigest(); expected=item['lfs']['sha256']
        else:
            h=hashlib.sha1(f"blob {item['size']}\0".encode())
            for chunk in iter(lambda:f.read(1048576),b''): h.update(chunk)
            actual=h.hexdigest(); expected=item['blobId']
    assert actual==expected, 'Pinned content mismatch: '+str(p)
# These byte hashes reproduce p1_download integrity; the frozen image identity manifest
# carries p1_audit hashrow's byte/decoded-RGB/capture checks, without rebuilding splits.
import csv
for row in csv.DictReader((ROOT/'data/manifests/shared/image_inventory.csv').open()):
    if row['group'] in groups:
        with (raw/row['image_path']).open('rb') as f: assert hashlib.file_digest(f,'sha256').hexdigest()==row['sha256']
if not (ROOT/'data/raw').exists(): (ROOT/'data/raw').symlink_to(raw,target_is_directory=True)
models=TEMP/'models';models.mkdir(exist_ok=True)
from urllib.request import urlretrieve
urls={'wide_resnet50_2-95faca4d.pth':'https://download.pytorch.org/models/wide_resnet50_2-95faca4d.pth','dinov2_vits14_pretrain.pth':'https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth'}
for name,url in urls.items():
    if not (models/name).exists(): urlretrieve(url,models/name)
    assert hashlib.sha256((models/name).read_bytes()).hexdigest()==cfg['asset_hashes'][name]
repo=models/'dinov2_repo'
if not repo.exists():
    archive=TEMP/'dinov2.zip';urlretrieve('https://codeload.github.com/facebookresearch/dinov2/zip/'+cfg['repo_commit'],archive)
    assert hashlib.sha256(archive.read_bytes()).hexdigest()=='04276715cddb29d45d05bff3a6fc132224dc27749b279ac98ad2ce4620e20d48'
    safe_extract(archive,models); (models/('dinov2-'+cfg['repo_commit'])).rename(repo)
for method, names in [('dinov2', ('dinov2_repo', 'dinov2_vits14_pretrain.pth')), ('patchcore', ('wide_resnet50_2-95faca4d.pth',))]:
    asset_dir = ROOT/'models'/method/'assets'
    asset_dir.mkdir(parents=True, exist_ok=True)
    for name in names:
        if not (asset_dir/name).exists():
            (asset_dir/name).symlink_to(models/name, target_is_directory=(models/name).is_dir())
print('Verified pinned dataset files:',len(files))


In [ ]:
freeze=json.loads((ROOT/'docs/freeze.json').read_text())
for key,name in [('protocol_hash','docs/protocol.md'),('config_hash','common/configs/config.yaml')]:
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest()==freeze[key], 'Freeze mismatch: '+name
sys.path.insert(0,str(ROOT))
from common.data import settings,load_roles,verify_assets
runtime=settings(ROOT/'common/configs/config.yaml');assert runtime['fingerprint']==freeze['fingerprint'];verify_assets(runtime)
for c in [f'test{i}' for i in range(1,7)]:
    for seed in range(5):
        for k in (0,5,20):load_roles(runtime,'MetalParts',c,k,seed)
        load_roles(runtime,'MetalParts',c,20,seed,True)
print('Freeze and 120 separation checks PASS')


In [ ]:
env=os.environ.copy();env['CUDA_VISIBLE_DEVICES']='-1'
result=subprocess.run([sys.executable,'common/run.py','--selftest'],cwd=ROOT,env=env,capture_output=True,text=True)
assert result.returncode==0,result.stderr[-2500:]
print('Synthetic CPU selftests PASS')


In [ ]:

def two_workers(arguments, timing=False):
    jobs = []
    deadline = SESSION_START + min(9, 12 - GPU_H_USED) * 3600
    try:
        for i in range(2):
            env = os.environ.copy()
            env['CUDA_VISIBLE_DEVICES'] = str(i)
            if timing:
                env['CACHE_DIR'] = str(TEMP / f'timing-{QUEUE}-{MODE}-{int(SESSION_START)}-{i}')
            log_path = OUT / f'run_shard{i}.log'
            with log_path.open('a', encoding='utf-8') as log:
                log.write(f"\n=== GPU {i}: {arguments} ===\n")
                log.flush()
                command = [sys.executable, '-u', 'common/run.py', '--device', 'cuda', '--shard', f'{i}/2', *arguments]
                process = subprocess.Popen(command, cwd=ROOT, env=env, stdout=log, stderr=subprocess.STDOUT)
            jobs.append((process, log_path))

        while True:
            codes = [process.poll() for process, _ in jobs]
            failures = [(code, log) for code, (_, log) in zip(codes, jobs) if code not in (None, 0)]
            if failures:
                for code, log in failures:
                    tail = log.read_text(encoding='utf-8', errors='replace').splitlines()[-32:]
                    print(f"{log.name} exited with code {code} for {arguments}:")
                    print("\n".join(tail))
                raise RuntimeError(f"Worker failed during {arguments}. Preserve any existing results ZIP.")
            if codes == [0, 0]:
                return
            if time.time() >= deadline:
                raise RuntimeError(f"Wall-clock cap reached during {arguments}. Preserve any existing results ZIP.")
            time.sleep(1)
    finally:
        active = [process for process, _ in jobs if process.poll() is None]
        for process in active:
            process.terminate()
        for process in active:
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
two_workers(['--timing'],timing=True)
measurements=[json.loads((OUT/f'timing.shard{i}.json').read_text()) for i in range(2)]
projection={k:max(x['queue_hours_estimate_two_shards'][k] for x in measurements) for k in ('Q1','Q2','Q3','Q4','session2')}
setup_h=(time.time()-SESSION_START)/3600
print('Projected hours (two shards, measured on this hardware):',projection)
print('Setup/preflight hours this run:',round(setup_h,4),'previous GPU hours:',GPU_H_USED)
needed=projection['Q1'] if QUEUE=='Q1' else projection['session2']
whole=projection['Q1']+projection['session2'] if QUEUE=='Q1' else projection['session2']
assert needed+setup_h<=9 and GPU_H_USED+setup_h+whole<=12, 'STOP: forecast exceeds budget; send projection for documented cuts (PCB, ablations, then seeds).'
if MODE=='timing': print('TIMING ONLY COMPLETE. Send this projection; full cells below are disabled.')


In [ ]:
def restore_q1(input_root, destination):
    archives = list(input_root.rglob('q1_resume.zip'))
    indexes = list(input_root.rglob('session_Q1_recovery.json'))
    if len(archives) + len(indexes) != 1:
        raise RuntimeError('Attach exactly one private Dataset containing q1_resume.zip or its extracted contents.')
    if archives:
        with archives[0].open('rb') as source:
            if hashlib.file_digest(source, 'sha256').hexdigest() != 'afdf315bf0d0bb639c032b358070423272d027e0442d659cb312dc7a135b756e':
                raise ValueError('Recovery ZIP checksum mismatch. Upload the original q1_resume.zip.')
        safe_extract(archives[0], destination)
    else:
        index = indexes[0]
        if hashlib.sha256(index.read_bytes()).hexdigest() != '2aa2ae2fecaceab4c02e2722ef978d512c6f77ca35de566f106b2b2b3069ea12':
            raise ValueError('Recovery manifest checksum mismatch. Upload the original q1_resume.zip.')
        manifest = json.loads(index.read_text(encoding='utf-8'))
        for entry in manifest['files']:
            relative = Path(entry['path'])
            source = index.parent / relative
            target = destination / relative
            if not source.resolve().is_relative_to(index.parent.resolve()) or not target.resolve().is_relative_to(destination.resolve()):
                raise ValueError(f'Unsafe recovery path: {relative}')
            with source.open('rb') as file:
                if hashlib.file_digest(file, 'sha256').hexdigest() != entry['sha256']:
                    raise ValueError(f'Recovery file checksum mismatch: {relative}')
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source, target)
        shutil.copyfile(index, destination / index.name)


if MODE=='full':
    if QUEUE == 'Q1' and Q1_RESUME:
        restore_q1(Path('/kaggle/input'), OUT)
        print('Restored 5 verified Q1 runs. The workers will skip them and run the remaining 265.')
    deadline=SESSION_START+min(9,12-GPU_H_USED)*3600;os.environ['DEADLINE_EPOCH']=str(deadline)
    (OUT/f'session_{QUEUE}.json').write_text(json.dumps({'queue':QUEUE,'start_epoch':SESSION_START,'previous_gpu_hours':GPU_H_USED,'deadline_epoch':deadline,'projection':projection,'freeze':freeze},indent=2))
    two_workers(['--extract'])
    for queue in (['Q1'] if QUEUE=='Q1' else ['Q2','Q3','Q4']):two_workers(['--queue',queue])


In [ ]:
if MODE=='full':
    import pandas as pd
    tables=[pd.read_csv(p) for p in (OUT/'raw').glob('*.shard*.csv')]; rows=pd.concat(tables,ignore_index=True)
    for q in (['Q1'] if QUEUE=='Q1' else ['Q2','Q3','Q4']):
        actual=rows[rows.queue==q]; expected=cfg['queue_sizes'][q]
        assert not actual.duplicated(['queue','job_index']).any(), 'Duplicate queue jobs'
        assert len(actual)==expected,(q,len(actual),expected)
        print(q,'expected',expected,'actual',len(actual),'per method',actual.groupby('method').size().to_dict())
    archive=OUT/f'results_{QUEUE}.zip';assert archive.is_file()
    state=OUT/f'session_{QUEUE}.json'; record=json.loads(state.read_text()); record.update(end_epoch=time.time(),wall_hours=(time.time()-SESSION_START)/3600);state.write_text(json.dumps(record,indent=2))
    temp=archive.with_suffix('.final.tmp')
    with zipfile.ZipFile(archive) as old, zipfile.ZipFile(temp,'w',zipfile.ZIP_STORED,allowZip64=True) as new:
        for info in old.infolist(): new.write(OUT/info.filename,info.filename)
    temp.replace(archive)
    with archive.open('rb') as f: print(archive.name,'SHA256',hashlib.file_digest(f,'sha256').hexdigest())
